# 1. Подготовка и методология

## 1.1. Методология расчета циклов восстановления (Recovery Time)

Для объективного анализа устойчивости валют мы используем концепцию **High-Water Mark (HWM)** и делим историю движения курса на завершенные и незавершенные циклы.

### Ключевые определения:
1. **High-Water Mark (HWM):** Исторический максимум абсолютного курса на временном отрезке от начала наблюдений до текущей даты.
2. **Состояние просадки (Under Water):** Период, когда текущий курс находится ниже зафиксированного ранее HWM.
3. **Цикл восстановления (Recovery Cycle):** Интервал времени, который включает три фазы:
    * **Пик (Peak):** Точка достижения нового HWM.
    * **Дно (Trough):** Точка максимального падения (MDD) внутри цикла.
    * **Восстановление (Recovery):** Точка первого возврата курса к уровню предыдущего HWM.

### Математические параметры:
* **Время падения (Drawdown Duration):** Количество календарных дней от *Пика* до *Дна*.
* **Время восстановления (Recovery Time):** Количество календарных дней от *Дна* до точки *Восстановления*.
* **Полный цикл:** Общее время нахождения «под водой» (от Пика до Восстановления).

#### Правила фильтрации и обработки данных:
* **Порог значимости:** Чтобы исключить технический шум, в анализе учитываются только циклы с просадкой (MDD) более **0.5%**.
* **Открытые просадки:** Если на текущую дату курс не вернулся к HWM, цикл считается «открытым». Для таких случаев рассчитывается текущее время нахождения в просадке, но они не включаются в расчет *среднего* времени восстановления, чтобы не занижать исторические показатели.
* **Абсолютные координаты:** Все расчеты производятся на базе абсолютных курсов [abscur.ru](https://www.abscur.ru), что исключает влияние циклов отдельных валют-якорей (например, USD).

## 1.2. Специфика абсолютных курсов: Очистка от «долларового шума»

Традиционный анализ валютных рисков через пары (например, USD/TRY или USD/EGP) страдает от фундаментального изъяна: **индикатор всегда зависит от состояния второй валюты (якоря).**

Если доллар США (USD) начинает цикл глобального укрепления из-за роста ставок ФРС, на графиках парных курсов кажется, что «падают» абсолютно все валюты мира. Это создает ложные сигналы о системном кризисе там, где его нет.

### В чем преимущество анализа просадок в абсолютных курсах:

1. **Изоляция локального риска:** Абсолютный курс рассчитывается относительно широкой корзины мировых валют. Если валюта страны X падает в абсолютном выражении, это означает реальный отток капитала из этой экономики, а не просто временную силу доллара.
2. **Истинное время восстановления:** В валютной паре актив может «восстановиться» только потому, что ослаб якорь (доллар), при этом реальная покупательная способность национальной валюты останется на дне. Абсолютный курс фиксирует восстановление только тогда, когда валюта реально возвращает свои позиции относительно всей мировой системы.
3. **Объективность циклов:** Использование данных [abscur.ru](https://www.abscur.ru) позволяет отделить глобальные макроэкономические циклы (связанные с эмиссией резервных валют) от индивидуальных циклов устойчивости каждой конкретной страны.

**Вывод:** Анализ времени восстановления в абсолютных координатах дает «чистую» картину жизнеспособности финансовой системы страны, исключая влияние внешних спекулятивных факторов.

# 2. Разработка алгоритма (Python)

## 2.1. Загрузка актуального датасета `abscur.csv`.

In [1]:
import pandas as pd

# Путь к файлу из подключенного датасета
file_path = '/kaggle/input/notebooks/eavprog/abscur2/abscur.csv'

# Загрузка данных с распознаванием дат
df = pd.read_csv(file_path, parse_dates=['Date'])

# Сортируем по дате (на случай, если в CSV порядок нарушен)
df = df.sort_values('Date')

# Устанавливаем дату как индекс — так удобнее делать срезы по времени (DateOffset)
df.set_index('Date', inplace=True)

# Проверка: выводим информацию о датасете и первые строки
print(f"Загружено строк: {len(df)}")
print(f"Диапазон дат: {df.index.min().date()} — {df.index.max().date()}")
print(f"Количество валют: {len(df.columns)}")

df.head()

Загружено строк: 5898
Диапазон дат: 2006-07-28 — 2026-03-16
Количество валют: 45


,AED,ARS,AUD,BRL,CAD,CHF,CLP,CNY,COP,CZK,...,SAR,SEK,SGD,THB,TRY,TWD,UAH,USD,VND,ZAR
Date,,,,,,,,,,,,,,,,,,,,,
2006-07-28,3.898841,1.683174,10.838947,5.596110,12.292434,15.511844,0.024427,1.827150,0.006823,0.649023,...,3.817157,1.655008,11.441231,0.436865,6.317193,0.443440,0.989630,14.320445,0.000673,1.282037
2006-07-31,3.898831,1.683169,10.838955,5.596095,12.292443,15.511855,0.024427,1.827266,0.006822,0.649024,...,3.817146,1.655010,11.441239,0.436864,6.317176,0.443439,0.989628,14.320406,0.000673,1.282038
2006-08-01,3.898863,1.683183,10.838931,5.596140,12.292415,15.511820,0.024427,1.826917,0.006823,0.649022,...,3.817177,1.655006,11.441214,0.436868,6.317227,0.443442,0.989636,14.320523,0.000673,1.282035
2006-08-02,3.898834,1.683171,10.838953,5.596099,12.292440,15.511852,0.024427,1.827231,0.006822,0.649024,...,3.817149,1.655009,11.441237,0.436864,6.317181,0.443439,0.989628,14.320418,0.000673,1.282038
2006-08-03,3.898884,1.683192,10.838915,5.596171,12.292397,15.511797,0.024427,1.826685,0.006823,0.649021,...,3.817198,1.655003,11.441197,0.436870,6.317262,0.443445,0.989641,14.320600,0.000673,1.282032


## 2.2. Реализация алгоритма поиска циклов: Пик -> Просадка -> Восстановление

Для автоматизации анализа мы разработали функцию, которая сканирует исторические данные каждой валюты и идентифицирует законченные и текущие периоды «нахождения под водой». 

**Логика работы функции:**
1. **Поиск HWM:** С помощью функции `cummax()` вычисляется скользящий максимум. Любое отклонение цены вниз от этой линии генерирует отрицательное значение просадки.
2. **Идентификация входа:** Когда просадка превышает заданный порог (0.5%), алгоритм фиксирует дату начала цикла и обновляет High-Water Mark.
3. **Фиксация дна:** В процессе падения алгоритм непрерывно отслеживает минимальную точку (Trough) и величину максимальной просадки (MDD) внутри конкретного цикла.
4. **Определение выхода:** Цикл считается завершенным, когда текущая цена касается или пересекает линию HWM. В этот момент фиксируется дата восстановления.
5. **Обработка текущих событий:** Если на последнюю дату датасета курс не восстановился, цикл помечается статусом `Ongoing` (в процессе), что критически важно для мониторинга актуальных рыночных рисков.

Этот подход позволяет превратить непрерывный временной ряд в структурированный набор событий, пригодных для глубокого статистического анализа.

In [2]:
import numpy as np

def get_recovery_stats(series, threshold=0.005):
    """
    Находит все циклы просадок для одной валюты.
    threshold: минимальная просадка (0.5%), чтобы игнорировать шум.
    """
    # Вычисляем накопленный максимум (High-Water Mark)
    hwm = series.cummax()
    
    # Считаем текущую просадку в процентах
    drawdowns = (series - hwm) / hwm
    
    cycles = []
    is_underwater = False
    start_date = None
    trough_date = None
    max_dd = 0
    
    for date, dd in drawdowns.items():
        if not is_underwater and dd < -threshold:
            # Начало новой просадки
            is_underwater = True
            start_date = date
            max_dd = dd
            trough_date = date
            
        elif is_underwater:
            # Обновляем максимальную просадку внутри цикла
            if dd < max_dd:
                max_dd = dd
                trough_date = date
            
            # Проверяем выход из просадки (вернулись к или превысили HWM)
            if dd >= 0:
                cycles.append({
                    'start': start_date,
                    'trough': trough_date,
                    'end': date,
                    'mdd': max_dd,
                    'recovery_days': (date - trough_date).days,
                    'total_days': (date - start_date).days,
                    'status': 'Recovered'
                })
                is_underwater = False
                max_dd = 0
                
    # Обрабатываем текущую (незавершенную) просадку
    if is_underwater:
        cycles.append({
            'start': start_date,
            'trough': trough_date,
            'end': series.index[-1],
            'mdd': max_dd,
            'recovery_days': (series.index[-1] - trough_date).days,
            'total_days': (series.index[-1] - start_date).days,
            'status': 'Ongoing'
        })
        
    return pd.DataFrame(cycles)

# Тестовый запуск для одной валюты (например, RUB)
test_currency = 'RUB'
rub_cycles = get_recovery_stats(df[test_currency])
print(f"Найдено циклов для {test_currency}: {len(rub_cycles)}")
rub_cycles.head()

Найдено циклов для RUB: 9


,start,trough,end,mdd,recovery_days,total_days,status
0,2007-08-15,2007-08-16,2007-09-20,-0.006169,35,36,Recovered
1,2007-12-14,2008-01-21,2008-03-06,-0.007236,45,83,Recovered
2,2008-05-07,2008-06-13,2008-07-15,-0.005463,32,69,Recovered
3,2008-08-05,2022-03-07,2022-05-12,-0.402849,66,5028,Recovered
4,2022-05-17,2022-05-17,2022-05-19,-0.012769,2,2,Recovered


## 2.3. Расчет длительности периодов «нахождения под водой»

На этом этапе мы переводим временные интервалы между ключевыми точками цикла (Пик, Дно, Восстановление) в конкретные календарные значения. 

**Что рассчитывается:**
* **Total Days (Общее время):** Количество календарных дней, которое валюта провела в состоянии просадки (от даты достижения максимума до даты возврата к нему).
* **Recovery Phase (Фаза восстановления):** Время, затраченное на путь от «дна» (максимальной просадки) до полного восстановления.

Эти метрики позволяют нам в дальнейшем классифицировать валюты не только по глубине падения, но и по скорости их реабилитации.

In [3]:
# Создаем список для хранения результатов по всем валютам
all_currency_cycles = []

for currency_name in df.columns:
    # Получаем циклы для текущей валюты
    res = get_recovery_stats(df[currency_name])
    res['currency'] = currency_name
    
    # Расчет длительности уже интегрирован в функцию get_recovery_stats, 
    # здесь мы приводим данные к итоговому виду для анализа
    all_currency_cycles.append(res)

# Объединяем в общую базу инцидентов
df_all_cycles = pd.concat(all_currency_cycles, ignore_index=True)

# Убеждаемся, что колонки длительности имеют целочисленный тип (дни)
df_all_cycles['total_days'] = df_all_cycles['total_days'].astype(int)
df_all_cycles['recovery_days'] = df_all_cycles['recovery_days'].astype(int)

# Выведем статистику по длительности для первых нескольких записей
print("Пример данных после расчета длительности (в днях):")
df_all_cycles[['currency', 'start', 'end', 'total_days', 'status']].head(10)

Пример данных после расчета длительности (в днях):


,currency,start,end,total_days,status
0,AED,2007-03-21,2008-10-22,581,Recovered
1,AED,2008-10-29,2008-11-19,21,Recovered
2,AED,2008-11-24,2009-02-26,94,Recovered
3,AED,2009-03-18,2015-01-02,2116,Recovered
4,AED,2015-02-03,2015-02-05,2,Recovered
5,AED,2015-02-13,2015-02-23,10,Recovered
6,AED,2015-02-25,2015-02-26,1,Recovered
7,AED,2015-02-27,2015-03-06,7,Recovered
8,AED,2015-03-18,2015-07-17,121,Recovered
9,AED,2015-09-11,2015-09-23,12,Recovered


## 2.4. Выделение и анализ «открытых просадок» (Ongoing Drawdowns)

Не все циклы в истории завершаются восстановлением. Некоторые валюты могут находиться в состоянии падения прямо сейчас. Обработка таких данных требует отдельного подхода, так как мы не знаем дату их завершения, но точно знаем, сколько дней они уже длятся.

**Зачем это нужно:**
1. **Актуальный риск:** Позволяет увидеть, какие валюты находятся в «затяжном пике» в данный момент.
2. **Корректность статистики:** Мы отделяем эти данные от исторических, чтобы среднее время восстановления считалось только по завершенным событиям, не искажая общую картину.

In [4]:
# Из общей базы выделяем только те записи, у которых статус 'Ongoing'
ongoing_drawdowns = df_all_cycles[df_all_cycles['status'] == 'Ongoing'].copy()

# Сортируем по длительности нахождения в просадке (от самых долгих к свежим)
ongoing_drawdowns = ongoing_drawdowns.sort_values('total_days', ascending=False)

# Выводим информацию о текущих проблемных активах
print(f"Количество валют в активной просадке: {len(ongoing_drawdowns)}")
print("\nТоп-10 самых длительных текущих просадок:")
ongoing_drawdowns[['currency', 'start', 'mdd', 'total_days']].head(10)

Количество валют в активной просадке: 42

Топ-10 самых длительных текущих просадок:


,currency,start,mdd,total_days
251,COP,2009-03-18,-0.401543,6207
1331,UAH,2009-03-18,-0.553422,6207
1266,TRY,2015-01-15,-0.923105,4078
93,BRL,2015-01-29,-0.460943,4064
45,ARS,2015-03-18,-0.991513,4016
751,KZT,2015-08-19,-0.554748,3862
351,EGP,2016-01-22,-0.805936,3706
955,PKR,2016-12-29,-0.570555,3364
1414,ZAR,2018-02-27,-0.271165,2939
199,CLP,2018-02-28,-0.279075,2938


# 3. Аналитическая часть

## 3.1. Расчет среднего времени восстановления (Average Recovery Time)

На этом этапе мы переходим от анализа отдельных инцидентов к системным показателям. Среднее время восстановления — это ключевой индикатор «живучести» финансовой системы. Он показывает, сколько в среднем инвестору приходится ждать возврата к безубыточности после падения курса в абсолютных координатах.

**Особенности реализации и нюансы расчета:**

1. **Исключение открытых просадок:** Для расчета среднего значения используются только завершенные циклы (`status == 'Recovered'`). Это критически важно: включение текущих (незавершенных) просадок искусственно занизило бы итоговый показатель, так как мы не знаем дату их фактического финала.
2. **Статистическая значимость:** Мы фокусируемся на валютах с достаточной историей наблюдений, чтобы среднее значение отражало характер валюты, а не случайный разовый инцидент.
3. **Интерактивная связь с [abscur.ru](https://www.abscur.ru):** Тикеры в итоговой таблице преобразованы в гиперссылки. При клике на тикер в новой вкладке открывается страница соответствующей валюты с живым графиком абсолютного курса. Это позволяет мгновенно сопоставить сухие цифры статистики с визуальной динамикой восстановления.

Такой подход превращает таблицу из простого списка в полноценный навигатор по валютным рискам.

In [5]:
from IPython.display import HTML

# Функция для создания HTML-ссылки на конкретную валюту
def make_clickable(ticker):
    url = f"https://www.abscur.ru/p/2.html?abs={ticker}"
    return f'<a href="{url}" target="_blank">{ticker}</a>'

# Фильтруем только завершенные циклы
recovered_only = df_all_cycles[df_all_cycles['status'] == 'Recovered']

# Группируем и считаем среднее
avg_recovery = recovered_only.groupby('currency')['total_days'].mean().reset_index()
avg_recovery['total_days'] = avg_recovery['total_days'].round(0).astype(int)
avg_recovery.columns = ['Currency', 'Avg_Recovery_Days']

# Сортируем
avg_recovery_sorted = avg_recovery.sort_values('Avg_Recovery_Days')

# Применяем создание ссылок к колонке Currency
# Создаем копию для отображения, чтобы не портить исходные данные для расчетов
display_df = avg_recovery_sorted.copy()
display_df['Currency'] = display_df['Currency'].apply(make_clickable)

print("Топ-10 валют с самым коротким средним временем восстановления (нажмите на тикер для перехода к графику):")
# Используем HTML() для отображения таблицы с работающими ссылками
HTML(display_df.head(10).to_html(escape=False, index=False))

Топ-10 валют с самым коротким средним временем восстановления (нажмите на тикер для перехода к графику):


Currency,Avg_Recovery_Days
TWD,100
CHF,109
HKD,113
CNY,120
PLN,123
SGD,127
NZD,131
KWD,132
THB,132
DKK,134


## 3.2. Выявление Max Recovery Time: Исторические периоды затяжных депрессий

Средние показатели часто сглаживают реальную картину рисков. Для глубокого понимания устойчивости необходимо знать «анти-рекорды» — максимальное время, которое потребовалось каждой валюте для выхода из самой тяжелой просадки в её истории.

**В этом разделе мы анализируем:**
* **Исторические депрессии:** Выявляем циклы, которые длились годами. Это те периоды, когда экономика страны сталкивалась с системными кризисами.
* **Интерактивный доступ:** Как и в предыдущем шаге, тикеры валют снабжены ссылками на [abscur.ru](https://www.abscur.ru), чтобы можно было визуально изучить структуру самого длительного падения.
* **Специфика данных:** В отличие от среднего, здесь мы смотрим на экстремум (`max`) по каждому активу, что позволяет оценить "запас терпения", который может потребоваться инвестору.

In [6]:
# Фильтруем только завершенные циклы (исторические анти-рекорды)
recovered_only = df_all_cycles[df_all_cycles['status'] == 'Recovered']

# Находим максимальное время восстановления для каждой валюты
max_recovery = recovered_only.groupby('currency')['total_days'].max().reset_index()

# Переименовываем колонки
max_recovery.columns = ['Currency', 'Max_Recovery_Days']

# Добавляем расчет в годах для наглядности (округляем до 1 знака)
max_recovery['Max_Recovery_Years'] = (max_recovery['Max_Recovery_Days'] / 365).round(1)

# Сортируем по убыванию — самые затяжные кризисы будут сверху
max_recovery_sorted = max_recovery.sort_values('Max_Recovery_Days', ascending=False)

# Подготавливаем интерактивную таблицу
display_max_df = max_recovery_sorted.copy()
display_max_df['Currency'] = display_max_df['Currency'].apply(make_clickable)

print("Топ-10 самых затяжных исторических депрессий (циклы, которые уже завершились):")
HTML(display_max_df.head(10).to_html(escape=False, index=False))

Топ-10 самых затяжных исторических депрессий (циклы, которые уже завершились):


Currency,Max_Recovery_Days,Max_Recovery_Years
RUB,5028,13.8
MXN,4975,13.6
NOK,4537,12.4
GBP,3944,10.8
EUR,3739,10.2
MYR,3220,8.8
AUD,3140,8.6
CAD,3001,8.2
ISK,2750,7.5
JPY,2708,7.4


## 3.3. Сводная таблица устойчивости: MDD vs Recovery Time

Это финальный этап аналитического блока, где мы объединяем два критических параметра риска. Глубина просадки (Maximum Drawdown) говорит нам о том, насколько «больно» падать, а время восстановления (Recovery Time) — насколько «долго» ждать возврата.

**Структура сводного отчета:**
* **Max Drawdown (MDD):** Худшее падение за всю историю наблюдений.
* **Avg Recovery:** Средний темп «зализывания ран» (в днях).
* **Max Recovery:** Худший исторический сценарий ожидания (в годах).
* **Ongoing Status:** Количество дней, если валюта находится в просадке прямо сейчас.

Такое сопоставление позволяет классифицировать валюты. Например, актив может иметь глубокие, но очень быстрые просадки, или наоборот — падать незначительно, но стагнировать десятилетиями.

In [7]:
# 1. Берем средние значения восстановления
summary = avg_recovery.copy()

# 2. Добавляем максимальное время восстановления (в годах для удобства)
summary = summary.merge(max_recovery[['Currency', 'Max_Recovery_Years']], on='Currency', how='left')

# 3. Добавляем максимальную глубину просадки (MDD) за всю историю из df_all_cycles
# Берем минимальное значение mdd (самое отрицательное)
mdd_stats = df_all_cycles.groupby('currency')['mdd'].min().reset_index()
mdd_stats['mdd'] = (mdd_stats['mdd'] * 100).round(2) # Переводим в проценты
mdd_stats.columns = ['Currency', 'Max_Drawdown_%']
summary = summary.merge(mdd_stats, on='Currency', how='left')

# 4. Добавляем текущий статус (сколько дней "под водой" сейчас)
ongoing_stats = ongoing_drawdowns[['currency', 'total_days']].copy()
ongoing_stats.columns = ['Currency', 'Current_Days_Underwater']
summary = summary.merge(ongoing_stats, on='Currency', how='left')

# Заполняем пропуски в текущих просадках нулями (если валюта на максимуме)
summary['Current_Days_Underwater'] = summary['Current_Days_Underwater'].fillna(0).astype(int)

# Сортируем по Max_Drawdown (от самых рисковых)
summary_sorted = summary.sort_values('Max_Drawdown_%')

# Финальное форматирование ссылок
display_summary = summary_sorted.copy()
display_summary['Currency'] = display_summary['Currency'].apply(make_clickable)

print("Сводная таблица валютных рисков (MDD vs Recovery):")
HTML(display_summary.to_html(escape=False, index=False))

Сводная таблица валютных рисков (MDD vs Recovery):


Currency,Avg_Recovery_Days,Max_Recovery_Years,Max_Drawdown_%,Current_Days_Underwater
ARS,409,5.8,-99.15,4016
TRY,706,5.8,-92.31,4078
EGP,448,5.8,-80.59,3706
PKR,258,5.7,-57.06,3364
KZT,301,5.9,-55.47,3862
UAH,232,1.6,-55.34,6207
BRL,708,5.9,-46.09,4064
RUB,656,13.8,-40.28,1355
COP,232,1.6,-40.15,6207
MXN,355,13.6,-28.93,703
